# WSN Intrusion Detection: Oversampling Method Validation and Real-World Performance Analysis

## Abstract

This notebook provides a comprehensive validation of oversampling methods for WSN intrusion detection, evaluating their effectiveness, real-world applicability, and adherence to best practices. We analyze synthetic data quality, model generalization, cross-validation performance, and real-world deployment considerations.

## Evaluation Framework

This evaluation encompasses:

1. **Synthetic Data Quality Assessment** - Evaluating the realism and diversity of generated samples
2. **Model Generalization Analysis** - Testing performance across different scenarios and datasets
3. **Cross-Validation Robustness** - Ensuring consistent performance across data splits
4. **Real-World Deployment Validation** - Assessing practical applicability and limitations
5. **Best Practices Compliance** - Verifying adherence to academic and industry standards
6. **Comparative Analysis** - Benchmarking against established methods and baselines

---

**Keywords**: Oversampling Validation, Synthetic Data Quality, Model Generalization, Real-World Performance

In [ ]:
# Library Initialization for Oversampling Validation
print("WSN Oversampling Validation Framework - Initialization")
print("=" * 55)

# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter
import os
import json
from datetime import datetime
from pathlib import Path

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, roc_curve
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
import joblib

# Specialized Libraries for Validation
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.stats import ks_2samp, wasserstein_distance

# Imbalanced Learning
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTEENN

# Suppress warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully")
print("✓ Validation framework ready")
print("=" * 55)

In [ ]:
# Data Loading and Preprocessing for Validation
print("Loading and preparing data for oversampling validation...")

# Load the original dataset
data_path = 'data/WSN-DS.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✓ Dataset loaded: {df.shape}")
else:
    print("❌ Dataset not found. Please ensure WSN-DS.csv is in the data folder.")

# Basic dataset information
print("\nDataset Overview:")
print(f"Shape: {df.shape}")
print(f"Features: {df.columns.tolist()}")
print(f"Target variable distribution:")
print(df['class'].value_counts())
print(f"Class distribution percentages:")
print(df['class'].value_counts(normalize=True) * 100)

# Data preprocessing (same as main notebook)
print("\nPreprocessing data...")

# Remove unnecessary columns if they exist
columns_to_drop = ['id', 'Time']
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
if existing_columns_to_drop:
    df = df.drop(columns=existing_columns_to_drop)
    print(f"✓ Dropped columns: {existing_columns_to_drop}")

# Handle missing values
if df.isnull().sum().any():
    print("Missing values found:")
    print(df.isnull().sum())
    df = df.dropna()
    print("✓ Missing values handled")
else:
    print("✓ No missing values found")

# Separate features and target
X = df.drop('class', axis=1)
y = df['class']

print(f"✓ Features shape: {X.shape}")
print(f"✓ Target shape: {y.shape}")
print(f"✓ Original class distribution: {Counter(y)}")

# Feature scaling for validation purposes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("✓ Data preprocessing completed successfully")

In [ ]:
# Oversampling Methods Implementation and Comparison
print("Implementing and comparing different oversampling methods...")
print("=" * 60)

# Split data for validation
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Original training class distribution: {Counter(y_train)}")

# Initialize oversampling methods
oversampling_methods = {
    'SMOTE': SMOTE(random_state=42),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

# Store results for comparison
oversampled_datasets = {}
oversampling_stats = {}

# Apply each oversampling method
for method_name, method in oversampling_methods.items():
    print(f"\nApplying {method_name}...")
    
    try:
        X_resampled, y_resampled = method.fit_resample(X_train, y_train)
        
        # Store the results
        oversampled_datasets[method_name] = {
            'X': X_resampled,
            'y': y_resampled
        }
        
        # Calculate statistics
        class_distribution = Counter(y_resampled)
        original_size = len(y_train)
        new_size = len(y_resampled)
        increase_ratio = new_size / original_size
        
        oversampling_stats[method_name] = {
            'original_size': original_size,
            'new_size': new_size,
            'increase_ratio': increase_ratio,
            'class_distribution': class_distribution,
            'minority_samples_added': new_size - original_size
        }
        
        print(f"✓ {method_name} completed")
        print(f"  Original size: {original_size}")
        print(f"  New size: {new_size}")
        print(f"  Increase ratio: {increase_ratio:.2f}x")
        print(f"  Class distribution: {class_distribution}")
        
    except Exception as e:
        print(f"❌ Error with {method_name}: {str(e)}")
        oversampling_stats[method_name] = {'error': str(e)}

print("\n✓ All oversampling methods applied successfully")
print("=" * 60)

In [ ]:
# Comprehensive Oversampling Quality Assessment
print("Evaluating quality of synthetic data generated by oversampling methods...")
print("=" * 70)

def calculate_data_quality_metrics(X_original, X_synthetic, method_name):
    """
    Calculate comprehensive quality metrics for synthetic data
    """
    print(f"\nEvaluating {method_name}...")
    
    quality_metrics = {}
    
    # 1. Statistical Distribution Comparison (Kolmogorov-Smirnov Test)
    ks_statistics = []
    ks_p_values = []
    
    for i, feature in enumerate(X_original.columns):
        original_feature = X_original.iloc[:, i]
        synthetic_feature = X_synthetic.iloc[:, i] if hasattr(X_synthetic, 'iloc') else X_synthetic[:, i]
        
        ks_stat, ks_p = ks_2samp(original_feature, synthetic_feature)
        ks_statistics.append(ks_stat)
        ks_p_values.append(ks_p)
    
    quality_metrics['ks_test'] = {
        'mean_ks_statistic': np.mean(ks_statistics),
        'mean_p_value': np.mean(ks_p_values),
        'features_significantly_different': sum([p < 0.05 for p in ks_p_values]),
        'total_features': len(ks_p_values)
    }
    
    # 2. Wasserstein Distance (Earth Mover's Distance)
    wasserstein_distances = []
    for i in range(X_original.shape[1]):
        original_feature = X_original.iloc[:, i] if hasattr(X_original, 'iloc') else X_original[:, i]
        synthetic_feature = X_synthetic.iloc[:, i] if hasattr(X_synthetic, 'iloc') else X_synthetic[:, i]
        
        wd = wasserstein_distance(original_feature, synthetic_feature)
        wasserstein_distances.append(wd)
    
    quality_metrics['wasserstein_distance'] = {
        'mean_distance': np.mean(wasserstein_distances),
        'std_distance': np.std(wasserstein_distances),
        'max_distance': np.max(wasserstein_distances)
    }
    
    # 3. Nearest Neighbor Analysis
    # Check if synthetic samples are too close to original samples (overfitting)
    if hasattr(X_synthetic, 'iloc'):
        X_synth_array = X_synthetic.values
    else:
        X_synth_array = X_synthetic
        
    if hasattr(X_original, 'iloc'):
        X_orig_array = X_original.values
    else:
        X_orig_array = X_original
    
    # Find nearest neighbors between synthetic and original data
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(X_orig_array)
    distances, indices = nn.kneighbors(X_synth_array)
    
    quality_metrics['nearest_neighbor'] = {
        'mean_distance_to_original': np.mean(distances),
        'std_distance_to_original': np.std(distances),
        'min_distance': np.min(distances),
        'potential_duplicates': sum(distances.flatten() < 1e-6)
    }
    
    # 4. Feature Correlation Preservation
    if hasattr(X_original, 'corr'):
        original_corr = X_original.corr()
    else:
        original_corr = pd.DataFrame(X_orig_array).corr()
        
    if hasattr(X_synthetic, 'corr'):
        synthetic_corr = X_synthetic.corr()
    else:
        synthetic_corr = pd.DataFrame(X_synth_array).corr()
    
    # Calculate correlation difference
    corr_diff = np.abs(original_corr.values - synthetic_corr.values)
    quality_metrics['correlation_preservation'] = {
        'mean_correlation_difference': np.mean(corr_diff[~np.isnan(corr_diff)]),
        'max_correlation_difference': np.max(corr_diff[~np.isnan(corr_diff)]),
        'correlation_similarity_score': 1 - np.mean(corr_diff[~np.isnan(corr_diff)])
    }
    
    # 5. Diversity Assessment
    # Calculate intra-class diversity
    synthetic_distances = pdist(X_synth_array)
    quality_metrics['diversity'] = {
        'mean_intra_synthetic_distance': np.mean(synthetic_distances),
        'std_intra_synthetic_distance': np.std(synthetic_distances),
        'diversity_coefficient': np.std(synthetic_distances) / np.mean(synthetic_distances) if np.mean(synthetic_distances) > 0 else 0
    }
    
    return quality_metrics

# Evaluate quality for each oversampling method
quality_results = {}

for method_name, dataset in oversampled_datasets.items():
    X_resampled = dataset['X']
    y_resampled = dataset['y']
    
    # Get only the minority class samples for evaluation
    minority_class = y_train.value_counts().idxmin()
    
    # Original minority samples
    X_original_minority = X_train[y_train == minority_class]
    
    # Synthetic minority samples (newly generated)
    original_minority_count = sum(y_train == minority_class)
    X_synthetic_minority = X_resampled[y_resampled == minority_class].iloc[original_minority_count:]
    
    if len(X_synthetic_minority) > 0:
        quality_metrics = calculate_data_quality_metrics(
            X_original_minority, X_synthetic_minority, method_name
        )
        quality_results[method_name] = quality_metrics
    else:
        print(f"No synthetic samples generated for {method_name}")

print("\n✓ Quality assessment completed for all methods")
print("=" * 70)

In [ ]:
# Quality Assessment Results Visualization
print("Creating comprehensive visualizations of oversampling quality...")
print("=" * 65)

# Create comprehensive quality comparison plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Oversampling Methods: Quality Assessment Comparison', fontsize=16, fontweight='bold')

methods = list(quality_results.keys())

# 1. KS Test Results
ks_means = [quality_results[method]['ks_test']['mean_ks_statistic'] for method in methods]
ks_p_values = [quality_results[method]['ks_test']['mean_p_value'] for method in methods]

axes[0, 0].bar(methods, ks_means, color='skyblue', alpha=0.7)
axes[0, 0].set_title('Mean KS Statistic\n(Lower = Better Distribution Match)')
axes[0, 0].set_ylabel('KS Statistic')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Wasserstein Distance
wd_means = [quality_results[method]['wasserstein_distance']['mean_distance'] for method in methods]

axes[0, 1].bar(methods, wd_means, color='lightgreen', alpha=0.7)
axes[0, 1].set_title('Mean Wasserstein Distance\n(Lower = Better Distribution Match)')
axes[0, 1].set_ylabel('Wasserstein Distance')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Nearest Neighbor Distance
nn_means = [quality_results[method]['nearest_neighbor']['mean_distance_to_original'] for method in methods]

axes[0, 2].bar(methods, nn_means, color='lightcoral', alpha=0.7)
axes[0, 2].set_title('Mean Distance to Original Data\n(Higher = Less Overfitting)')
axes[0, 2].set_ylabel('Distance')
axes[0, 2].tick_params(axis='x', rotation=45)

# 4. Correlation Preservation
corr_sim = [quality_results[method]['correlation_preservation']['correlation_similarity_score'] for method in methods]

axes[1, 0].bar(methods, corr_sim, color='gold', alpha=0.7)
axes[1, 0].set_title('Correlation Similarity Score\n(Higher = Better Preservation)')
axes[1, 0].set_ylabel('Similarity Score')
axes[1, 0].tick_params(axis='x', rotation=45)

# 5. Diversity Assessment
diversity_coeff = [quality_results[method]['diversity']['diversity_coefficient'] for method in methods]

axes[1, 1].bar(methods, diversity_coeff, color='mediumpurple', alpha=0.7)
axes[1, 1].set_title('Diversity Coefficient\n(Higher = More Diverse)')
axes[1, 1].set_ylabel('Diversity Coefficient')
axes[1, 1].tick_params(axis='x', rotation=45)

# 6. Overall Quality Score (Composite)
# Calculate composite quality score (normalize and combine metrics)
quality_scores = {}
for method in methods:
    # Normalize metrics (0-1 scale)
    ks_score = 1 - (quality_results[method]['ks_test']['mean_ks_statistic'] / max(ks_means))
    wd_score = 1 - (quality_results[method]['wasserstein_distance']['mean_distance'] / max(wd_means))
    nn_score = quality_results[method]['nearest_neighbor']['mean_distance_to_original'] / max(nn_means)
    corr_score = quality_results[method]['correlation_preservation']['correlation_similarity_score']
    div_score = quality_results[method]['diversity']['diversity_coefficient'] / max(diversity_coeff) if max(diversity_coeff) > 0 else 0
    
    # Weighted composite score
    composite_score = (0.25 * ks_score + 0.25 * wd_score + 0.2 * nn_score + 
                      0.2 * corr_score + 0.1 * div_score)
    quality_scores[method] = composite_score

quality_values = list(quality_scores.values())
axes[1, 2].bar(methods, quality_values, color='orange', alpha=0.7)
axes[1, 2].set_title('Composite Quality Score\n(Higher = Better Overall)')
axes[1, 2].set_ylabel('Quality Score')
axes[1, 2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print detailed quality assessment results
print("\nDetailed Quality Assessment Results:")
print("=" * 50)

for method in methods:
    print(f"\n{method}:")
    print(f"  KS Test - Mean Statistic: {quality_results[method]['ks_test']['mean_ks_statistic']:.4f}")
    print(f"  KS Test - Mean P-value: {quality_results[method]['ks_test']['mean_p_value']:.4f}")
    print(f"  Wasserstein Distance: {quality_results[method]['wasserstein_distance']['mean_distance']:.4f}")
    print(f"  Distance to Original: {quality_results[method]['nearest_neighbor']['mean_distance_to_original']:.4f}")
    print(f"  Correlation Similarity: {quality_results[method]['correlation_preservation']['correlation_similarity_score']:.4f}")
    print(f"  Diversity Coefficient: {quality_results[method]['diversity']['diversity_coefficient']:.4f}")
    print(f"  Composite Quality Score: {quality_scores[method]:.4f}")
    print(f"  Potential Duplicates: {quality_results[method]['nearest_neighbor']['potential_duplicates']}")

# Rank methods by quality
ranked_methods = sorted(quality_scores.items(), key=lambda x: x[1], reverse=True)
print(f"\nRanking by Composite Quality Score:")
print("=" * 40)
for i, (method, score) in enumerate(ranked_methods, 1):
    print(f"{i}. {method}: {score:.4f}")

print("\n✓ Quality assessment visualization completed")
print("=" * 65)

In [ ]:
# Real-World Performance Validation
print("Evaluating real-world performance of oversampling methods...")
print("=" * 60)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import make_scorer, f1_score

# Define multiple classifiers for robust evaluation
classifiers = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42, probability=True),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500)
}

# Performance metrics to evaluate
metrics = {
    'accuracy': accuracy_score,
    'f1_macro': lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro'),
    'f1_weighted': lambda y_true, y_pred: f1_score(y_true, y_pred, average='weighted')
}

# Store performance results
performance_results = {}

# Evaluate each oversampling method with each classifier
for method_name, dataset in oversampled_datasets.items():
    print(f"\nEvaluating {method_name}...")
    
    X_resampled = dataset['X']
    y_resampled = dataset['y']
    
    performance_results[method_name] = {}
    
    for clf_name, classifier in classifiers.items():
        print(f"  Testing with {clf_name}...")
        
        try:
            # Cross-validation evaluation
            cv_scores = {}
            
            for metric_name, metric_func in metrics.items():
                scorer = make_scorer(metric_func)
                cv_score = cross_val_score(
                    classifier, X_resampled, y_resampled, 
                    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                    scoring=scorer, n_jobs=-1
                )
                cv_scores[metric_name] = {
                    'mean': cv_score.mean(),
                    'std': cv_score.std(),
                    'scores': cv_score.tolist()
                }
            
            # Train on resampled data and test on original test set
            classifier.fit(X_resampled, y_resampled)
            y_pred = classifier.predict(X_test)
            
            # Calculate test performance
            test_scores = {}
            for metric_name, metric_func in metrics.items():
                test_scores[metric_name] = metric_func(y_test, y_pred)
            
            performance_results[method_name][clf_name] = {
                'cv_scores': cv_scores,
                'test_scores': test_scores
            }
            
            print(f"    CV F1-macro: {cv_scores['f1_macro']['mean']:.4f} (±{cv_scores['f1_macro']['std']:.4f})")
            print(f"    Test F1-macro: {test_scores['f1_macro']:.4f}")
            
        except Exception as e:
            print(f"    Error: {str(e)}")
            performance_results[method_name][clf_name] = {'error': str(e)}

# Baseline performance (no oversampling)
print(f"\nEvaluating Baseline (No Oversampling)...")
baseline_results = {}

for clf_name, classifier in classifiers.items():
    print(f"  Testing with {clf_name}...")
    
    try:
        # Cross-validation on original training data
        cv_scores = {}
        
        for metric_name, metric_func in metrics.items():
            scorer = make_scorer(metric_func)
            cv_score = cross_val_score(
                classifier, X_train, y_train,
                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                scoring=scorer, n_jobs=-1
            )
            cv_scores[metric_name] = {
                'mean': cv_score.mean(),
                'std': cv_score.std(),
                'scores': cv_score.tolist()
            }
        
        # Train and test
        classifier.fit(X_train, y_train)
        y_pred = classifier.predict(X_test)
        
        test_scores = {}
        for metric_name, metric_func in metrics.items():
            test_scores[metric_name] = metric_func(y_test, y_pred)
        
        baseline_results[clf_name] = {
            'cv_scores': cv_scores,
            'test_scores': test_scores
        }
        
        print(f"    CV F1-macro: {cv_scores['f1_macro']['mean']:.4f} (±{cv_scores['f1_macro']['std']:.4f})")
        print(f"    Test F1-macro: {test_scores['f1_macro']:.4f}")
        
    except Exception as e:
        print(f"    Error: {str(e)}")
        baseline_results[clf_name] = {'error': str(e)}

print("\n✓ Real-world performance evaluation completed")
print("=" * 60)

In [ ]:
# Performance Comparison Visualization and Analysis
print("Creating comprehensive performance comparison visualizations...")
print("=" * 65)

# Extract performance data for visualization
def extract_performance_data():
    """Extract and organize performance data for visualization"""
    
    # Initialize data structures
    methods_list = ['Baseline'] + list(oversampled_datasets.keys())
    classifiers_list = list(classifiers.keys())
    
    # Create matrices for heatmaps
    f1_macro_cv = np.zeros((len(methods_list), len(classifiers_list)))
    f1_macro_test = np.zeros((len(methods_list), len(classifiers_list)))
    accuracy_test = np.zeros((len(methods_list), len(classifiers_list)))
    
    # Fill matrices
    for i, method in enumerate(methods_list):
        for j, clf_name in enumerate(classifiers_list):
            if method == 'Baseline':
                results = baseline_results
            else:
                results = performance_results[method]
            
            if clf_name in results and 'error' not in results[clf_name]:
                f1_macro_cv[i, j] = results[clf_name]['cv_scores']['f1_macro']['mean']
                f1_macro_test[i, j] = results[clf_name]['test_scores']['f1_macro']
                accuracy_test[i, j] = results[clf_name]['test_scores']['accuracy']
            else:
                f1_macro_cv[i, j] = np.nan
                f1_macro_test[i, j] = np.nan
                accuracy_test[i, j] = np.nan
    
    return methods_list, classifiers_list, f1_macro_cv, f1_macro_test, accuracy_test

methods_list, classifiers_list, f1_macro_cv, f1_macro_test, accuracy_test = extract_performance_data()

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Oversampling Methods: Real-World Performance Comparison', fontsize=16, fontweight='bold')

# 1. F1-Macro Cross-Validation Scores Heatmap
im1 = axes[0, 0].imshow(f1_macro_cv, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[0, 0].set_title('F1-Macro Score (Cross-Validation)')
axes[0, 0].set_xticks(range(len(classifiers_list)))
axes[0, 0].set_xticklabels(classifiers_list, rotation=45)
axes[0, 0].set_yticks(range(len(methods_list)))
axes[0, 0].set_yticklabels(methods_list)

# Add text annotations
for i in range(len(methods_list)):
    for j in range(len(classifiers_list)):
        if not np.isnan(f1_macro_cv[i, j]):
            text = axes[0, 0].text(j, i, f'{f1_macro_cv[i, j]:.3f}',
                                 ha="center", va="center", color="black", fontsize=8)

plt.colorbar(im1, ax=axes[0, 0])

# 2. F1-Macro Test Scores Heatmap
im2 = axes[0, 1].imshow(f1_macro_test, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[0, 1].set_title('F1-Macro Score (Test Set)')
axes[0, 1].set_xticks(range(len(classifiers_list)))
axes[0, 1].set_xticklabels(classifiers_list, rotation=45)
axes[0, 1].set_yticks(range(len(methods_list)))
axes[0, 1].set_yticklabels(methods_list)

# Add text annotations
for i in range(len(methods_list)):
    for j in range(len(classifiers_list)):
        if not np.isnan(f1_macro_test[i, j]):
            text = axes[0, 1].text(j, i, f'{f1_macro_test[i, j]:.3f}',
                                 ha="center", va="center", color="black", fontsize=8)

plt.colorbar(im2, ax=axes[0, 1])

# 3. Average Performance Comparison
avg_f1_cv = np.nanmean(f1_macro_cv, axis=1)
avg_f1_test = np.nanmean(f1_macro_test, axis=1)

x = np.arange(len(methods_list))
width = 0.35

axes[1, 0].bar(x - width/2, avg_f1_cv, width, label='CV F1-Macro', alpha=0.7)
axes[1, 0].bar(x + width/2, avg_f1_test, width, label='Test F1-Macro', alpha=0.7)
axes[1, 0].set_title('Average F1-Macro Performance')
axes[1, 0].set_xlabel('Methods')
axes[1, 0].set_ylabel('F1-Macro Score')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(methods_list, rotation=45)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Performance Improvement Analysis
baseline_avg = avg_f1_test[0]  # Baseline is first in the list
improvements = avg_f1_test[1:] - baseline_avg  # Exclude baseline

axes[1, 1].bar(methods_list[1:], improvements, 
               color=['green' if x > 0 else 'red' for x in improvements], alpha=0.7)
axes[1, 1].set_title('Performance Improvement over Baseline')
axes[1, 1].set_xlabel('Oversampling Methods')
axes[1, 1].set_ylabel('F1-Macro Improvement')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)

plt.tight_layout()
plt.show()

# Detailed Performance Analysis
print("\nDetailed Performance Analysis:")
print("=" * 50)

print(f"\nBaseline Performance (No Oversampling):")
for clf_name in classifiers_list:
    if clf_name in baseline_results and 'error' not in baseline_results[clf_name]:
        cv_f1 = baseline_results[clf_name]['cv_scores']['f1_macro']['mean']
        test_f1 = baseline_results[clf_name]['test_scores']['f1_macro']
        test_acc = baseline_results[clf_name]['test_scores']['accuracy']
        print(f"  {clf_name}: CV F1={cv_f1:.4f}, Test F1={test_f1:.4f}, Test Acc={test_acc:.4f}")

print(f"\nOversampling Methods Performance:")
for method in oversampled_datasets.keys():
    print(f"\n{method}:")
    method_f1_scores = []
    for clf_name in classifiers_list:
        if (clf_name in performance_results[method] and 
            'error' not in performance_results[method][clf_name]):
            cv_f1 = performance_results[method][clf_name]['cv_scores']['f1_macro']['mean']
            test_f1 = performance_results[method][clf_name]['test_scores']['f1_macro']
            test_acc = performance_results[method][clf_name]['test_scores']['accuracy']
            method_f1_scores.append(test_f1)
            print(f"  {clf_name}: CV F1={cv_f1:.4f}, Test F1={test_f1:.4f}, Test Acc={test_acc:.4f}")
    
    if method_f1_scores:
        avg_improvement = np.mean(method_f1_scores) - baseline_avg
        print(f"  Average F1 Improvement: {avg_improvement:+.4f}")

# Best performing combinations
print(f"\nBest Performing Combinations:")
print("=" * 35)

best_combinations = []
for i, method in enumerate(methods_list):
    for j, clf_name in enumerate(classifiers_list):
        if not np.isnan(f1_macro_test[i, j]):
            best_combinations.append((method, clf_name, f1_macro_test[i, j]))

# Sort by performance
best_combinations.sort(key=lambda x: x[2], reverse=True)

for i, (method, clf, score) in enumerate(best_combinations[:10]):
    print(f"{i+1:2d}. {method} + {clf}: {score:.4f}")

print("\n✓ Performance comparison analysis completed")
print("=" * 65)